In [1]:
import sempy.fabric as fabric
import time, json, base64

client = fabric.FabricRestClient()
workspace_id = fabric.get_workspace_id()

# Step 1: Create or get existing Env
try:
    response = client.post(f"/v1/workspaces/{workspace_id}/environments", json={"displayName": "Env"})
    env_id = response.json()["id"]
    print(f"Created Environment: {env_id}")
except Exception as e:
    envs = client.get(f"/v1/workspaces/{workspace_id}/environments").json()
    env_id = next(env["id"] for env in envs["value"] if env["displayName"] == "Env")
    print(f"Found existing Environment: {env_id}")

# Step 2: Upload libraries
yml_content = """dependencies:
  - pip:
    - semantic-link-labs
    - fabric-data-agent-sdk==0.1.9a0
    - tabulate==0.9.0
"""
lib_response = client.post(
    f"/v1/workspaces/{workspace_id}/environments/{env_id}/staging/libraries",
    files={"file": ("environment.yml", yml_content.encode(), "text/plain")}
)
print(f"Libraries uploaded: {lib_response.status_code}")

# Step 3: Publish and WAIT until fully published
publish_response = client.post(f"/v1/workspaces/{workspace_id}/environments/{env_id}/staging/publish")
print(f"Publish triggered: {publish_response.status_code}")

print("Waiting for publish to complete...")
while True:
    time.sleep(15)
    env_details = client.get(f"/v1/workspaces/{workspace_id}/environments/{env_id}").json()
    state = env_details.get("properties", {}).get("publishDetails", {}).get("state", "")
    print(f"  Publish state: {state}")
    if state == "Success":
        print("✅ Environment published!")
        break
    elif state in ["Failed", "Cancelled"]:
        raise Exception(f"Publish failed: {env_details}")

# Step 4: Attach to all notebooks
print("\nAttaching Env to all notebooks...")
items = client.get(f"/v1/workspaces/{workspace_id}/items?type=Notebook").json()

for item in items["value"]:
    try:
        # Get definition
        def_resp = client.post(f"/v1/workspaces/{workspace_id}/items/{item['id']}/getDefinition")
        
        if def_resp.status_code == 202:
            location = def_resp.headers.get("Location")
            while True:
                time.sleep(5)
                if client.get(location).json().get("status") == "Succeeded":
                    break
            def_json = client.get(location + "/result").json()
        else:
            def_json = def_resp.json()

        parts = def_json.get("definition", {}).get("parts", [])
        updated = False

        for part in parts:
            # Update .platform part
            if part["path"] == ".platform":
                content = json.loads(base64.b64decode(part["payload"]).decode("utf-8"))
                content.setdefault("config", {})
                content["config"]["environmentId"] = env_id
                content["config"]["workspaceId"] = workspace_id
                part["payload"] = base64.b64encode(json.dumps(content).encode()).decode()
                updated = True

            # Also update .ipynb metadata
            if part["path"].endswith(".ipynb"):
                content = json.loads(base64.b64decode(part["payload"]).decode("utf-8"))
                content.setdefault("metadata", {}).setdefault("trident", {})["environment"] = {
                    "environmentId": env_id,
                    "workspaceId": workspace_id
                }
                part["payload"] = base64.b64encode(json.dumps(content).encode()).decode()
                updated = True

        if updated:
            update_resp = client.post(
                f"/v1/workspaces/{workspace_id}/items/{item['id']}/updateDefinition",
                json={"definition": {"parts": parts}}
            )
            status = "✅ attached" if update_resp.status_code in [200, 202] else f"❌ error {update_resp.status_code}"
            print(f"  {status}: {item['displayName']}")

    except Exception as ex:
        print(f"  ❌ error on {item['displayName']}: {ex}")

print("\nDone!")

StatementMeta(, 0d224629-6387-4517-923f-1c3c8ad407b0, 3, Finished, Available, Finished, False)

Created Environment: 95309176-8132-4398-b73c-62e11683a2fd
Libraries uploaded: 200
Publish triggered: 200
Waiting for publish to complete...
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Running
  Publish state: Success
✅ Environment published!

Attaching